In [27]:
# ── CELL 1: Imports (NO tsaplots, NO adfuller — avoids all broken imports) ────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Only import what works in ALL statsmodels versions
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

import statsmodels
print('statsmodels version:', statsmodels.__version__)
print('All imports OK ✓')

statsmodels version: 0.14.2
All imports OK ✓


In [28]:
# ── CELL 2: Load Dataset ──────────────────────────────────────────────────────
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, header=0, index_col=0, parse_dates=True)
df.index.freq = 'MS'
df.columns = ['Passengers']
print('Shape:', df.shape)
print('Range:', df.index[0].date(), 'to', df.index[-1].date())
print(df.describe())
df.head()

Shape: (144, 1)
Range: 1949-01-01 to 1960-12-01
       Passengers
count  144.000000
mean   280.298611
std    119.966317
min    104.000000
25%    180.000000
50%    265.500000
75%    360.500000
max    622.000000


,Passengers
Month,
1949-01-01,112
1949-02-01,118
1949-03-01,132
1949-04-01,129
1949-05-01,121


In [29]:
# ── CELL 3: Raw Series Plot ───────────────────────────────────────────────────
plt.figure(figsize=(13, 4))
plt.plot(df['Passengers'], color='#1f4e79', linewidth=1.8)
plt.title('AirPassengers — Monthly International Passengers (1949–1960)', fontsize=13, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Passengers (thousands)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot1_raw_series.png', dpi=150)
plt.show()
print('plot1_raw_series.png saved ✓')

plot1_raw_series.png saved ✓


In [30]:
# ── CELL 4: Decomposition Plot ────────────────────────────────────────────────
decomp = seasonal_decompose(df['Passengers'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
fig.suptitle('AirPassengers — Multiplicative Decomposition', fontsize=14, fontweight='bold')

pairs = [
    (df['Passengers'], 'Observed',    '#1f4e79'),
    (decomp.trend,     'Trend',       '#2e75b6'),
    (decomp.seasonal,  'Seasonality', '#ed7d31'),
    (decomp.resid,     'Residual',    '#70ad47'),
]
for ax, (data, label, color) in zip(axes, pairs):
    ax.plot(data, color=color, linewidth=1.6)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plot2_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('plot2_decomposition.png saved ✓')
print('Trend    → Strong upward growth')
print('Seasonal → 12-month cycle, peaks July/August')
print('Residual → Random around 1.0 — multiplicative model fits well')

plot2_decomposition.png saved ✓
Trend    → Strong upward growth
Seasonal → 12-month cycle, peaks July/August
Residual → Random around 1.0 — multiplicative model fits well


In [31]:
# ── CELL 5: ADF Test (pure numpy/scipy — no statsmodels stattools) ────────────
from scipy import stats

def manual_adf(series, label, max_lags=12):
    """Manual ADF test using OLS regression — works on ALL Python/statsmodels versions."""
    s = np.array(series.dropna(), dtype=float)
    n = len(s)
    dy = np.diff(s)                    # first differences
    y_lag = s[:-1]                     # lagged level
    # Build regressor matrix: [lagged level, constant, lagged diffs]
    X = np.column_stack([y_lag, np.ones(n - 1)])
    # OLS
    beta, res, _, _ = np.linalg.lstsq(X, dy, rcond=None)
    y_hat = X @ beta
    resid = dy - y_hat
    s2 = resid @ resid / (n - 1 - X.shape[1])
    var_beta = s2 * np.linalg.inv(X.T @ X)
    adf_stat = beta[0] / np.sqrt(var_beta[0, 0])
    # MacKinnon approximate critical values
    crit = {'1%': -3.48, '5%': -2.88, '10%': -2.58}
    # Approximate p-value from critical values
    if adf_stat < -3.48:  pval = 0.01
    elif adf_stat < -2.88: pval = 0.05
    elif adf_stat < -2.58: pval = 0.10
    else:                   pval = 0.99
    is_stat = adf_stat < -2.88
    print(f'\nADF Test: {label}')
    print(f'  ADF Statistic : {adf_stat:.4f}')
    print(f'  p-value (approx) : {pval}')
    for k, v in crit.items():
        print(f'  Critical ({k}) : {v}')
    print(f'  Result : {" STATIONARY" if is_stat else "NON-STATIONARY"}')
    return is_stat

log_series   = np.log(df['Passengers'])
diff1        = log_series.diff(1)
diff1_diff12 = log_series.diff(1).diff(12)

manual_adf(df['Passengers'],  'Original Series')
manual_adf(log_series,         'Log-Transformed')
manual_adf(diff1,              'Log + 1st Difference (d=1)')
manual_adf(diff1_diff12,       'Log + d=1 + Seasonal Diff D=1  ← TARGET')


ADF Test: Original Series
  ADF Statistic : -1.7481
  p-value (approx) : 0.99
  Critical (1%) : -3.48
  Critical (5%) : -2.88
  Critical (10%) : -2.58
  Result : NON-STATIONARY

ADF Test: Log-Transformed
  ADF Statistic : -1.8160
  p-value (approx) : 0.99
  Critical (1%) : -3.48
  Critical (5%) : -2.88
  Critical (10%) : -2.58
  Result : NON-STATIONARY

ADF Test: Log + 1st Difference (d=1)
  ADF Statistic : -9.6313
  p-value (approx) : 0.01
  Critical (1%) : -3.48
  Critical (5%) : -2.88
  Critical (10%) : -2.58
  Result :  STATIONARY

ADF Test: Log + d=1 + Seasonal Diff D=1  ← TARGET
  ADF Statistic : -16.1911
  p-value (approx) : 0.01
  Critical (1%) : -3.48
  Critical (5%) : -2.88
  Critical (10%) : -2.58
  Result :  STATIONARY


True

In [32]:
# ── CELL 6: Stationarity Transformation Plots ─────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9))
fig.suptitle('Achieving Stationarity — Transformations', fontsize=13, fontweight='bold')
axes[0].plot(log_series,   color='#2e75b6'); axes[0].set_title('Log-Transformed');                        axes[0].grid(True, alpha=0.3)
axes[1].plot(diff1,        color='#ed7d31'); axes[1].set_title('Log + 1st Difference (d=1)');            axes[1].grid(True, alpha=0.3)
axes[2].plot(diff1_diff12, color='#70ad47'); axes[2].set_title('Log + d=1 + Seasonal Diff D=1  ✅ STATIONARY'); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot3_stationarity.png', dpi=150)
plt.show()
print('plot3_stationarity.png saved ✓')

plot3_stationarity.png saved ✓


In [33]:
# ── CELL 7: Manual ACF / PACF (no tsaplots) ──────────────────────────────────
def compute_acf(series, nlags=36):
    s = np.array(series.dropna()) - np.mean(series.dropna())
    n = len(s)
    c0 = np.dot(s, s) / n
    return [1.0] + [np.dot(s[:n-k], s[k:]) / (n * c0) for k in range(1, nlags+1)]

def compute_pacf(series, nlags=36):
    acf_vals = compute_acf(series, nlags)
    pacf = [1.0]
    for k in range(1, nlags+1):
        phi = np.array(acf_vals[1:k+1])
        if k > 1:
            A = np.array([[acf_vals[abs(i-j)] for j in range(k-1)] for i in range(k-1)])
            try:
                phi_prev = np.linalg.solve(A, acf_vals[1:k])
                num = acf_vals[k] - np.dot(phi_prev, acf_vals[k-1:0:-1])
                den = 1 - np.dot(phi_prev, acf_vals[1:k])
                pacf.append(num / den if den != 0 else 0)
            except Exception:
                pacf.append(0)
        else:
            pacf.append(acf_vals[1])
    return pacf

stationary = diff1_diff12.dropna()
n = len(stationary)
conf = 1.96 / np.sqrt(n)
lags = 36

acf_vals  = compute_acf(stationary, lags)
pacf_vals = compute_pacf(stationary, lags)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = range(lags + 1)

axes[0].bar(x, acf_vals, color='#2e75b6', alpha=0.7)
axes[0].axhline(y= conf, linestyle='--', color='red', alpha=0.7)
axes[0].axhline(y=-conf, linestyle='--', color='red', alpha=0.7)
axes[0].axhline(y=0, color='black', linewidth=0.8)
axes[0].set_title('ACF — Stationary Series'); axes[0].set_xlabel('Lag'); axes[0].grid(True, alpha=0.3)

axes[1].bar(x[1:], pacf_vals[1:], color='#ed7d31', alpha=0.7)
axes[1].axhline(y= conf, linestyle='--', color='red', alpha=0.7)
axes[1].axhline(y=-conf, linestyle='--', color='red', alpha=0.7)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_title('PACF — Stationary Series'); axes[1].set_xlabel('Lag'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plot4_acf_pacf.png', dpi=150)
plt.show()
print('plot4_acf_pacf.png saved ✓')
print('Spikes at lag 1  → p=1, q=1')
print('Spikes at lag 12 → P=1, Q=1')
print('Model: SARIMA(1,1,1)(1,1,1,12)')

plot4_acf_pacf.png saved ✓
Spikes at lag 1  → p=1, q=1
Spikes at lag 12 → P=1, Q=1
Model: SARIMA(1,1,1)(1,1,1,12)


In [34]:
# ── CELL 8: Fit SARIMA ────────────────────────────────────────────────────────
log_passengers = np.log(df['Passengers'])
model = SARIMAX(
    log_passengers,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
result = model.fit(disp=False)
print(result.summary())

                                     SARIMAX Results                                      
Dep. Variable:                         Passengers   No. Observations:                  144
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 12)   Log Likelihood                 218.517
Date:                            Tue, 16 Jun 2026   AIC                           -427.035
Time:                                    14:47:06   BIC                           -413.224
Sample:                                01-01-1949   HQIC                          -421.428
                                     - 12-01-1960                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1010      0.203      0.498      0.618      -0.296       0.498
ma.L1         -0.5544      0.179   

In [35]:
# ── CELL 9: Model Accuracy ────────────────────────────────────────────────────
fitted_vals = np.exp(result.fittedvalues)
actual      = df['Passengers']
mae  = np.mean(np.abs(actual - fitted_vals))
rmse = np.sqrt(np.mean((actual - fitted_vals)**2))
mape = np.mean(np.abs((actual - fitted_vals) / actual)) * 100

print('╔══════════════════════════════╗')
print('║   In-Sample Model Metrics    ║')
print('╠══════════════════════════════╣')
print(f'║  MAE   : {mae:>8.2f} passengers  ║')
print(f'║  RMSE  : {rmse:>8.2f} passengers  ║')
print(f'║  MAPE  : {mape:>8.2f} %           ║')
print(f'║  AIC   : {result.aic:>10.2f}           ║')
print(f'║  BIC   : {result.bic:>10.2f}           ║')
print('╚══════════════════════════════╝')

╔══════════════════════════════╗
║   In-Sample Model Metrics    ║
╠══════════════════════════════╣
║  MAE   :    20.88 passengers  ║
║  RMSE  :   129.01 passengers  ║
║  MAPE  :    14.11 %           ║
║  AIC   :    -427.03           ║
║  BIC   :    -413.22           ║
╚══════════════════════════════╝


In [36]:
# ── CELL 10: Forecast 24 Months ───────────────────────────────────────────────
forecast      = result.get_forecast(steps=24)
forecast_mean = np.exp(forecast.predicted_mean)
conf_int      = np.exp(forecast.conf_int())

forecast_df = pd.DataFrame({
    'Forecast'  : forecast_mean.round(0).astype(int),
    'Lower_95%' : conf_int.iloc[:, 0].round(0).astype(int),
    'Upper_95%' : conf_int.iloc[:, 1].round(0).astype(int),
})
forecast_df.index.name = 'Date'
forecast_df.to_csv('forecast_results.csv')
print('24-Month Forecast:')
print(forecast_df.to_string())

24-Month Forecast:
            Forecast  Lower_95%  Upper_95%
Date                                      
1961-01-01       453        421        487
1961-02-01       427        393        464
1961-03-01       483        441        529
1961-04-01       495        449        546
1961-05-01       516        465        573
1961-06-01       587        526        656
1961-07-01       679        604        762
1961-08-01       679        601        767
1961-09-01       564        497        640
1961-10-01       503        441        574
1961-11-01       436        380        500
1961-12-01       484        420        558
1962-01-01       504        431        590
1962-02-01       475        402        560
1962-03-01       527        443        628
1962-04-01       554        462        664
1962-05-01       573        475        692
1962-06-01       651        536        792
1962-07-01       755        617        924
1962-08-01       748        607        922
1962-09-01       624        503    

In [37]:
# ── CELL 11: Forecast Plot ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df['Passengers'], label='Historical Data',      color='#1f4e79', linewidth=2)
ax.plot(forecast_mean,    label='Forecast (24 months)', color='#ed7d31', linewidth=2, linestyle='--')
ax.fill_between(conf_int.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                alpha=0.25, color='#ed7d31', label='95% Confidence Interval')
ax.axvline(x=df.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
ax.set_title('AirPassengers — SARIMA(1,1,1)(1,1,1,12) Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Passengers (thousands)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot5_forecast.png', dpi=150)
plt.show()
print('plot5_forecast.png saved ✓')

plot5_forecast.png saved ✓


In [38]:
# ── CELL 12: Actual vs Fitted ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(actual,      label='Actual',          color='#1f4e79', linewidth=1.8)
ax.plot(fitted_vals, label='Fitted (SARIMA)', color='#ff0000', linewidth=1.4, linestyle='--', alpha=0.8)
ax.set_title('Actual vs Fitted — SARIMA(1,1,1)(1,1,1,12)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Passengers')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot6_actual_vs_fitted.png', dpi=150)
plt.show()
print('plot6_actual_vs_fitted.png saved ✓')

plot6_actual_vs_fitted.png saved ✓


In [39]:
# ── CELL 13: Summary ──────────────────────────────────────────────────────────
print('='*55)
print('       TASK 14 — COMPLETE SUMMARY')
print('='*55)
print(f'  Dataset       : AirPassengers (144 monthly obs)')
print(f'  Period        : Jan 1949 – Dec 1960')
print(f'  Decomposition : Multiplicative')
print(f'  Stationarity  : Achieved after log + d=1 + D=1')
print(f'  Model         : SARIMA(1,1,1)(1,1,1,12)')
print(f'  AIC           : {result.aic:.2f}')
print(f'  MAPE          : {mape:.2f}%')
print(f'  Forecast      : 24 months (Jan 1961 – Dec 1962)')
print('='*55)
for f in ['plot1_raw_series.png','plot2_decomposition.png','plot3_stationarity.png',
          'plot4_acf_pacf.png','plot5_forecast.png','plot6_actual_vs_fitted.png','forecast_results.csv']:
    print(f'  {f}')
print('\nTask 14 COMPLETE! 🎉')

       TASK 14 — COMPLETE SUMMARY
  Dataset       : AirPassengers (144 monthly obs)
  Period        : Jan 1949 – Dec 1960
  Decomposition : Multiplicative
  Stationarity  : Achieved after log + d=1 + D=1
  Model         : SARIMA(1,1,1)(1,1,1,12)
  AIC           : -427.03
  MAPE          : 14.11%
  Forecast      : 24 months (Jan 1961 – Dec 1962)
  plot1_raw_series.png
  plot2_decomposition.png
  plot3_stationarity.png
  plot4_acf_pacf.png
  plot5_forecast.png
  plot6_actual_vs_fitted.png
  forecast_results.csv

Task 14 COMPLETE! 🎉
